# Training Models (iBVPNet & FactorizePhys) for Group F

This notebook contains the training pipeline for iBVPNet and FactorizePhys using the preprocessed Group F dataset.

In [ ]:
import os
import sys
import glob
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Setup REPO_ROOT
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.iBVPNet import iBVPNet
from neural_methods.model.FactorizePhys.FactorizePhys import FactorizePhys
from neural_methods.loss.NegPearsonLoss import Neg_Pearson


In [ ]:
# ----- Configs -----
PREPROCESSED_PATH = os.path.join(REPO_ROOT, "preprocessed_data/Headmotion/groupF")
OUTPUT_DIR = os.path.join(REPO_ROOT, "final_model_release")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_LENGTH = 160
BATCH_SIZE = 4
EPOCHS = 10
LR = 1e-3

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


In [ ]:
# ----- Dataset & DataLoader -----
class GroupFDataset(Dataset):
    def __init__(self, preprocessed_dir):
        # Use all available input clips across all subjects
        self.inputs = sorted(glob.glob(os.path.join(preprocessed_dir, "*", "*_input*.npy")))
        self.labels = [f.replace("input", "label") for f in self.inputs]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data = np.float32(np.load(self.inputs[index]))   # (T, H, W, 3)
        label = np.float32(np.load(self.labels[index]))  # (T,)

        # NCDHW -> (C, T, H, W)
        data = np.transpose(data, (3, 0, 1, 2))
        
        fname      = os.path.basename(self.inputs[index])
        try:
            split_idx  = fname.index("_")
            subject_id = fname[:split_idx]
            chunk_id   = fname[split_idx + 6:].split(".")[0]
        except ValueError:
            subject_id = "unknown"
            chunk_id = "0"

        return data, label, subject_id, chunk_id

dataset = GroupFDataset(PREPROCESSED_PATH)
print(f"Total clips found: {len(dataset)}")

# Using the entire dataset for training
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
print(f"DataLoader configured with {len(train_loader)} batches.")


## 1. Train iBVPNet

In [ ]:
def train_ibvpnet():
    print("\n--- Training iBVPNet ---")
    model = iBVPNet(frames=CHUNK_LENGTH, in_channels=3).to(DEVICE)
    criterion = Neg_Pearson()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader)
    )

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        tbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for batch in tbar:
            data = batch[0].to(DEVICE)     # (N, C, T, H, W)
            labels = batch[1].to(DEVICE)   # (N, T)
            
            # Frame padding for internal torch.diff usage
            last_frame = data[:, :, -1:, :, :].clone()
            data_padded = torch.cat([data, last_frame], dim=2)
            
            pred_ppg = model(data_padded)
            
            # Normalization
            pred_ppg = (pred_ppg - torch.mean(pred_ppg)) / (torch.std(pred_ppg) + 1e-8)
            labels = (labels - torch.mean(labels)) / (torch.std(labels) + 1e-8)
            
            loss = criterion(pred_ppg, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            running_loss += loss.item()
            tbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        print(f"Epoch {epoch+1} Avg Loss: {running_loss / len(train_loader):.4f}")
        
    save_path = os.path.join(OUTPUT_DIR, "GroupF_iBVPNet.pth")
    torch.save(model.state_dict(), save_path)
    print(f"Saved iBVPNet to {save_path}")


In [ ]:
# train_ibvpnet()


## 2. Train FactorizePhys

In [ ]:
def train_factorizephys():
    print("\n--- Training FactorizePhys ---")
    MD_CONFIG = {
        "FRAME_NUM": CHUNK_LENGTH,
        "MD_FSAM": True,
        "MD_TYPE": "NMF",
        "MD_TRANSFORM": "T_KAB",
        "MD_R": 1,
        "MD_S": 1,
        "MD_STEPS": 3,
        "MD_INFERENCE": False,  # False for training
        "MD_RESIDUAL": True,
    }
    
    model = FactorizePhys(
        frames=CHUNK_LENGTH, 
        md_config=MD_CONFIG, 
        in_channels=3, 
        dropout=0.1, 
        device=torch.device(DEVICE)
    ).to(DEVICE)
    
    criterion = Neg_Pearson()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader)
    )

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        tbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for batch in tbar:
            data = batch[0].to(DEVICE)
            labels = batch[1].to(DEVICE)
            
            # Optional: handle multi-signal labelled data if it's 3D
            if len(labels.shape) > 2:
                labels = labels[..., 0]
                
            # Frame padding
            last_frame = data[:, :, -1:, :, :].clone()
            data_padded = torch.cat([data, last_frame], dim=2)
            
            # Forward pass
            # With MD_FSAM=True and model in train mode, returns 4 outputs
            pred_ppg, vox_embed, factorized_embed, appx_error = model(data_padded)
            
            # Normalization
            pred_ppg = (pred_ppg - torch.mean(pred_ppg)) / (torch.std(pred_ppg) + 1e-8)
            labels = (labels - torch.mean(labels)) / (torch.std(labels) + 1e-8)
            
            loss = criterion(pred_ppg, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            running_loss += loss.item()
            tbar.set_postfix({"loss": f"{loss.item():.4f}", "appx_err": f"{appx_error.item():.4f}"})
            
        print(f"Epoch {epoch+1} Avg Loss: {running_loss / len(train_loader):.4f}")
        
    save_path = os.path.join(OUTPUT_DIR, "GroupF_FactorizePhys.pth")
    torch.save(model.state_dict(), save_path)
    print(f"Saved FactorizePhys to {save_path}")


In [ ]:
# train_factorizephys()
